# GroupDNA — WhatsApp Group Chat Analyzer

## Minor Project — Week 1 Python Fundamentals

**Name:** V.Swapnith Chowdary  
**Project:** GroupDNA  
**Dataset:** `hostel_bois.txt`  
**Date:** 21 September 2026

### Objective

To build a Python-based WhatsApp group chat analyzer that reads a raw `.txt` chat export, parses the messages, analyzes group activity and communication patterns, creates a NumPy-based activity heatmap, identifies frequently used words, measures response and silent patterns, and assigns a personality archetype to each participant.

### Technologies / Concepts Used
- Python fundamentals
- Strings, Lists, Tuples, Sets and Dictionaries
- Loops and Conditional Statements
- Functions and f-strings
- File I/O
- `datetime`
- NumPy

> **AI-assisted:** This notebook was organized and formatted with AI assistance. The final code should be reviewed and understood by the student before submission.


# 1. Problem Statement

We are given a raw WhatsApp chat export in a plaintext `.txt` file. The task is to read the file, parse the messages, and produce a comprehensive analytical report about the group.

The report includes:
1. Chat parsing
2. Group overview
3. Busiest day and hour
4. NumPy activity heatmap
5. Top words
6. Response speed and silent days
7. Personality archetype detection
8. Final formatted report


# 2. Understanding the Input File

The expected WhatsApp export format is:

`DD/MM/YY, HH:MM - Sender Name: Message text`

The parser also handles:
- System messages
- Media omitted messages
- Deleted messages
- Empty lines
- Timestamp conversion using `datetime`


In [ ]:
# ============================================================
# GROUPDNA - WHATSAPP GROUP CHAT ANALYZER
# ============================================================

import numpy as np
from datetime import datetime


# ============================================================


# 3. Feature 1 — Chat Parser

Reads `hostel_bois.txt`, separates the timestamp, sender and message text, handles special cases, converts timestamps to `datetime`, and stores valid messages in a list of dictionaries.


In [2]:
# FEATURE 1: READ AND PARSE CHAT FILE
# ============================================================

from datetime import datetime

file_path = "/content/hostel_bois.txt"

messages = []

system_count = 0
media_count = 0
deleted_count = 0

with open(file_path, "r", encoding="utf-8") as file:

    lines = file.readlines()


for line in lines:

    line = line.strip()

    # Skip empty lines
    if line == "":
        continue

    # Try to separate timestamp and remaining text
    try:
        timestamp, remaining = line.split(" - ", 1)
        sender, text = remaining.split(": ", 1)

    except ValueError:

        # System message or invalid line
        system_count += 1
        continue

    # Media message
    if text == "<Media omitted>":
        media_count += 1
        continue

    # Deleted message
    if text == "This message was deleted":
        deleted_count += 1
        continue

    # Store valid message
    message = {
        "timestamp": timestamp,
        "sender": sender,
        "text": text
    }

    # Convert timestamp to datetime
    try:
        message["datetime"] = datetime.strptime(
            timestamp,
            "%d/%m/%y, %H:%M"
        )
    except ValueError:
        continue

    messages.append(message)


print("Messages parsed:", len(messages))
print("System messages:", system_count)
print("Media messages:", media_count)
print("Deleted messages:", deleted_count)


# ============================================================


Messages parsed: 3127
System messages: 4
Media messages: 32
Deleted messages: 15


# 4. Feature 2 — Group Overview

Calculates the participants, total messages, date range, total days, and messages sent by each participant.


In [3]:
# FEATURE 2: GROUP OVERVIEW
# ============================================================

participants = set()
person_count = {}

for message in messages:

    person = message["sender"]

    participants.add(person)

    if person not in person_count:
        person_count[person] = 0

    person_count[person] += 1


# Sort people according to message count
sorted_people = sorted(
    person_count.items(),
    key=lambda x: x[1],
    reverse=True
)


# Date range
dates = []

for message in messages:
    dates.append(message["datetime"].date())


first_date = min(dates)
last_date = max(dates)

total_days = (last_date - first_date).days + 1


print()
print("=" * 65)
print("GROUP OVERVIEW")
print("=" * 65)

print("Group         : Hostel Bois 4ever")
print(
    f"Period        : {first_date.strftime('%d %B %Y')} "
    f"to {last_date.strftime('%d %B %Y')}"
)
print("Total days    :", total_days)
print("Total messages:", len(messages))
print("Participants  :", len(participants))

print()
print("MESSAGES PER PERSON")

for person, count in sorted_people:

    percentage = (count / len(messages)) * 100

    bar = "█" * int(percentage / 2)

    print(
        f"{person:<10} "
        f"{bar:<20} "
        f"{count:>4} "
        f"({percentage:>5.1f}%)"
    )


# ============================================================



GROUP OVERVIEW
Group         : Hostel Bois 4ever
Period        : 01 April 2024 to 30 May 2024
Total days    : 60
Total messages: 3127
Participants  : 6

MESSAGES PER PERSON
Rahul      ███████████████       940 ( 30.1%)
Priya      ███████████           712 ( 22.8%)
Neha       █████████             624 ( 20.0%)
Aman       ███████               484 ( 15.5%)
Karan      █████                 345 ( 11.0%)
Vikas                             22 (  0.7%)


# 5. Feature 3 — Busiest Day and Hour

Finds the day with the highest number of messages and the hour with the highest overall message activity.


In [4]:
# FEATURE 3: BUSIEST DAY AND HOUR
# ============================================================

day_count = {}
hour_count = {}

for message in messages:

    date = message["datetime"].date()
    hour = message["datetime"].hour

    # Count days
    if date not in day_count:
        day_count[date] = 0

    day_count[date] += 1

    # Count hours
    if hour not in hour_count:
        hour_count[hour] = 0

    hour_count[hour] += 1


busiest_day = max(
    day_count,
    key=day_count.get
)

busiest_hour = max(
    hour_count,
    key=hour_count.get
)


print()
print("=" * 65)
print("GROUP ACTIVITY")
print("=" * 65)

print(
    f"Busiest day  : "
    f"{busiest_day.strftime('%d %B %Y')} "
    f"({day_count[busiest_day]} messages)"
)

print(
    f"Busiest hour : "
    f"{busiest_hour:02d}:00 - "
    f"{(busiest_hour + 1) % 24:02d}:00 "
    f"({hour_count[busiest_hour]} messages)"
)


# ============================================================



GROUP ACTIVITY
Busiest day  : 04 May 2024 (74 messages)
Busiest hour : 18:00 - 19:00 (244 messages)


# 6. Feature 4 — NumPy Activity Heatmap

Creates a participant × hour matrix using NumPy. Each row represents a participant and each column represents an hour from 00 to 23. The matrix is displayed as a text-based heatmap.


In [14]:
# FEATURE 4: NUMPY ACTIVITY HEATMAP
# ============================================================

import numpy as np

people = sorted(participants)

# Create 6 x 24 matrix
heatmap = np.zeros(
    (len(people), 24),
    dtype=int
)


# Give every person a row number
person_index = {}

for index, person in enumerate(people):

    person_index[person] = index


# Fill the matrix
for message in messages:

    person = message["sender"]
    hour = message["datetime"].hour

    row = person_index[person]

    heatmap[row, hour] += 1


print()
print("=" * 65)
print("ACTIVITY HEATMAP")
print("=" * 65)

print("Person     ", end="")

for hour in range(24):
    print(f"{hour:02d} ", end="")

print()


for index, person in enumerate(people):
    row = heatmap[index]
    maximum = row.max()
    print(f"{person:<10} ", end="")
    for value in row:
        if maximum == 0:
            symbol = "."
        else:
            ratio = value / maximum
            if ratio < 0.25:
                symbol = "."
            elif ratio < 0.50:
                symbol = "░"
            elif ratio < 0.75:
                symbol = "▒"
            else:
                symbol = "█"
        print(symbol + "  ", end="")
    print()
# ============================================================



ACTIVITY HEATMAP
Person     00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23 
Aman       ▒  █  █  ▒  █  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  ▒  
Karan      .  .  .  .  .  .  .  .  ░  ░  ▒  ░  █  ▒  █  ▒  █  █  ▒  █  ▒  ░  ░  .  
Neha       .  .  .  .  .  ░  .  .  ▒  █  █  ░  ▒  ▒  ░  .  ▒  █  █  █  ▒  ░  ░  ░  
Priya      .  .  .  .  .  .  .  ░  ▒  █  █  █  █  █  ▒  ░  ▒  ▒  ▒  █  ▒  ▒  ░  .  
Rahul      .  .  .  .  .  .  .  .  .  .  .  .  ▒  ░  ░  ▒  ▒  ░  █  ▒  ░  █  ▒  ▒  
Vikas      .  .  .  .  .  .  .  ░  ▒  ░  ░  .  ░  ▒  .  ░  ░  █  ▒  ▒  ░  ░  ░  ▒  


# 7. Feature 5 — Top Words

Counts words across real messages, converts them to lowercase, removes punctuation and ignores common stop words. The top 10 words are displayed with simple text bars.


In [6]:
# FEATURE 5: TOP WORDS
# ============================================================
stop_words = {
    "i",
    "is",
    "the",
    "a",
    "and",
    "or",
    "to",
    "of",
    "in",
    "on",
    "for",
    "it",
    "my",
    "me",
    "you",
    "we",
    "are",
    "was",
    "this",
    "that"
}
word_count = {}
for message in messages:
    text = message["text"].lower()
    words = text.split()
    for word in words:
        # Remove punctuation
        word = word.strip(
            ".,!?;:\"'()[]{}<>"
        )
        if word == "":
            continue
        if word in stop_words:
            continue
        if word not in word_count:
            word_count[word] = 0
        word_count[word] += 1
top_words = sorted(
    word_count.items(),
    key=lambda x: x[1],
    reverse=True
)
print()
print("=" * 65)
print("THIS GROUP'S FAVOURITE WORDS")
print("=" * 65)
for word, count in top_words[:10]:
    bar_length = min(count // 10, 25)
    bar = "█" * bar_length
    print(
        f"{word:<12} "
        f"{bar:<25} "
        f"{count}"
    )
# ============================================================



THIS GROUP'S FAVOURITE WORDS
how          █████████████████████████ 321
guys         █████████████████████████ 318
so           █████████████████████████ 292
about        █████████████████████████ 274
hai          █████████████████████████ 268
am           █████████████████████████ 260
today        █████████████████████████ 257
at           █████████████████████████ 257
he           ██████████████████████    220
his          █████████████████████     217


# 8. Feature 6 — Message Length

Calculates the total and average number of words per message for each participant. These values are later used by the personality-archetype analysis.


In [9]:
# FEATURE 6: MESSAGE LENGTH
# ============================================================
person_words = {}
person_messages = {}
for message in messages:
    person = message["sender"]
    words = message["text"].split()
    if person not in person_words:
        person_words[person] = 0
        person_messages[person] = 0
    person_words[person] += len(words)
    person_messages[person] += 1
average_words = {}
for person in people:
    average_words[person] = (
        person_words[person]
        / person_messages[person]
    )
# ============================================================


## Feature 6A — Response Time

Calculates the time gap between a message from one participant and the next message from a different participant, then finds the average response time for each person.


In [10]:
# FEATURE 6A: RESPONSE TIME
# ============================================================
response_gaps = {}
for person in people:
    response_gaps[person] = []
for i in range(1, len(messages)):
    previous = messages[i - 1]
    current = messages[i]
    # Only calculate when a different person replies
    if previous["sender"] != current["sender"]:
        person = current["sender"]
        gap = (
            current["datetime"]
            - previous["datetime"]
        )
        minutes = gap.total_seconds() / 60
        # Ignore extremely large gaps
        if minutes >= 0:
            response_gaps[person].append(
                minutes
            )
average_response = {}
for person in people:
    if len(response_gaps[person]) > 0:
        average_response[person] = (
            sum(response_gaps[person])
            / len(response_gaps[person])
        )
    else:
        average_response[person] = 0
fastest_person = min(
    average_response,
    key=average_response.get
)
slowest_person = max(
    average_response,
    key=average_response.get
)
print()
print("=" * 65)
print("RESPONSE PATTERNS")
print("=" * 65)

print(
    f"Fastest replier : "
    f"{fastest_person} "
    f"({average_response[fastest_person]:.1f} minutes)"
)
print(
    f"Slowest replier : "
    f"{slowest_person} "
    f"({average_response[slowest_person] / 60:.1f} hours)"
)
# ============================================================



RESPONSE PATTERNS
Fastest replier : Vikas (34.9 minutes)
Slowest replier : Aman (0.9 hours)


## Feature 6B — Silent Days

Calculates how many days in the chat period each participant did not send any messages.


In [11]:
# FEATURE 6B: SILENT DAYS
# ============================================================
all_dates = set(dates)
active_days = {}
for person in people:
    active_days[person] = set()
    for message in messages:
        if message["sender"] == person:
            active_days[person].add(
                message["datetime"].date()
            )
silent_days = {}
for person in people:
    silent_days[person] = (
        len(all_dates)
        - len(active_days[person])
    )
print()
print("SILENT DAYS")
for person in sorted(
    people,
    key=lambda x: silent_days[x],
    reverse=True
):
    print(
        f"{person:<10} "
        f"{silent_days[person]} days"
    )
# ============================================================



SILENT DAYS
Vikas      44 days
Aman       0 days
Karan      0 days
Neha       0 days
Priya      0 days
Rahul      0 days


# 9. Feature 7 — Personality Archetype Detection

The project uses quantitative rules to calculate personality-related scores for each participant.

The archetypes supported by the project are:
- The Spammer
- The Group Mom
- The Night Owl
- The Storyteller
- The Drama Queen
- The Ghost
- The Comedian
- The Question Master


In [15]:
# FEATURE 7: PERSONALITY SCORES
# ============================================================
# -----------------------------
# NIGHT OWL
# -----------------------------

night_percentage = {}
for person in people:
    total = 0
    night = 0
    for message in messages:
        if message["sender"] == person:
            total += 1

            message_hour = message["datetime"].hour

            if message_hour >= 23 or message_hour <= 4:
                night += 1

    if total > 0:

        night_percentage[person] = (
            night / total
        ) * 100

    else:

        night_percentage[person] = 0


# -----------------------------
# DRAMA QUEEN
# -----------------------------

drama_percentage = {}

for person in people:

    total = 0
    drama = 0

    for message in messages:

        if message["sender"] != person:
            continue

        text = message["text"].strip()

        total += 1

        if len(text) >= 3:

            if (
                text.isupper()
                or text.count("!") >= 2
            ):

                drama += 1

    if total > 0:

        drama_percentage[person] = (
            drama / total
        ) * 100

    else:

        drama_percentage[person] = 0


# -----------------------------
# QUESTION MASTER
# -----------------------------

question_percentage = {}

for person in people:

    total = 0
    questions = 0

    for message in messages:

        if message["sender"] == person:

            total += 1

            if message["text"].strip().endswith("?"):

                questions += 1

    if total > 0:

        question_percentage[person] = (
            questions / total
        ) * 100

    else:

        question_percentage[person] = 0


# -----------------------------
# COMEDIAN
# -----------------------------

funny_words = {
    "lol",
    "lmao",
    "haha",
    "rofl",
    "lmfao"
}

comedian_score = {}

for person in people:

    total_messages = 0
    funny_count = 0

    for message in messages:

        if message["sender"] != person:
            continue

        total_messages += 1

        words = (
            message["text"]
            .lower()
            .split()
        )

        for word in words:

            word = word.strip(
                ".,!?;:\"'()[]{}"
            )

            if word in funny_words:

                funny_count += 1

    if total_messages > 0:

        comedian_score[person] = (
            funny_count / total_messages
        ) * 100

    else:

        comedian_score[person] = 0


# ============================================================


## Feature 7A — Spammer Score

Measures consecutive message bursts from each participant.


In [16]:
# FEATURE 7A: SPAMMER SCORE
# ============================================================

burst_scores = {}

for person in people:

    bursts = []
    current_burst = 0

    previous_sender = None

    for message in messages:

        sender = message["sender"]

        if sender == person:

            if previous_sender == person:

                current_burst += 1

            else:

                if current_burst > 0:
                    bursts.append(current_burst)

                current_burst = 1

        else:

            if current_burst > 0:

                bursts.append(current_burst)

                current_burst = 0

        previous_sender = sender

    if current_burst > 0:
        bursts.append(current_burst)

    if len(bursts) > 0:

        burst_scores[person] = (
            sum(bursts) / len(bursts)
        )

    else:

        burst_scores[person] = 0


# ============================================================


## Feature 7B — Group Mom Score

Counts caring or reminder-related keywords in each participant's messages.


In [19]:
# FEATURE 7B: GROUP MOM
# ============================================================

caring_words = [
    "okay",
    "safe",
    "eat",
    "sleep",
    "take care",
    "are you",
    "please",
    "reminder",
    "drink water",
    "don't forget"
]

group_mom_score = {}

for person in people:

    score = 0

    for message in messages:

        if message["sender"] != person:
            continue

        text = message["text"].lower()

        for word in caring_words:

            if word in text:

                score += 1

    group_mom_score[person] = score


# ============================================================


## Feature 7C — Create Final Archetype Scores

Combines all calculated scores and assigns the highest-scoring archetype to each participant.


In [20]:
# FEATURE 7C: CREATE ARCHETYPE SCORES
# ============================================================

archetypes = {}

for person in people:

    scores = {}

    # Spammer
    scores["THE SPAMMER"] = (
        burst_scores[person]
    )

    # Group Mom
    scores["THE GROUP MOM"] = (
        group_mom_score[person]
    )

    # Night Owl
    scores["THE NIGHT OWL"] = (
        night_percentage[person]
    )

    # Storyteller
    scores["THE STORYTELLER"] = (
        average_words[person]
    )

    # Drama Queen
    scores["THE DRAMA QUEEN"] = (
        drama_percentage[person]
    )

    # Ghost
    silent_percent = (
        silent_days[person]
        / total_days
    ) * 100

    scores["THE GHOST"] = silent_percent

    # Comedian
    scores["THE COMEDIAN"] = (
        comedian_score[person]
    )

    # Question Master
    scores["THE QUESTION MASTER"] = (
        question_percentage[person]
    )

    # Select highest score
    archetypes[person] = max(
        scores,
        key=scores.get
    )


# ============================================================


# 10. Feature 8 — Final Report

Combines the major findings into one clean, formatted GroupDNA report suitable for viewing in the notebook output.


In [ ]:
# FEATURE 8: FINAL REPORT
# ============================================================

print()
print()
print("=" * 65)
print("              GROUPDNA REPORT")
print("=" * 65)

print(
    f"{total_days} days • "
    f"{len(messages)} messages • "
    f"{len(participants)} members"
)

print("=" * 65)

print()
print("GROUP OVERVIEW")

print(
    f"Period       : "
    f"{first_date.strftime('%d %B %Y')} "
    f"to "
    f"{last_date.strftime('%d %B %Y')}"
)

print(
    f"Busiest day  : "
    f"{busiest_day.strftime('%d %B %Y')} "
    f"({day_count[busiest_day]} messages)"
)

print(
    f"Busiest hour : "
    f"{busiest_hour:02d}:00 - "
    f"{(busiest_hour + 1) % 24:02d}:00"
)


print()
print("MESSAGES PER PERSON")

for person, count in sorted_people:

    percentage = (
        count / len(messages)
    ) * 100

    bar = "█" * int(percentage / 2)

    print(
        f"{person:<10} "
        f"{bar:<20} "
        f"{count} "
        f"({percentage:.1f}%)"
    )


print()
print("TOP WORDS")

for word, count in top_words[:5]:

    bar = "█" * min(count // 10, 20)

    print(
        f"{word:<12} "
        f"{bar:<20} "
        f"{count}"
    )


print()
print("RESPONSE PATTERNS")

print(
    f"Fastest replier : "
    f"{fastest_person} "
    f"({average_response[fastest_person]:.1f} min)"
)

print(
    f"Slowest replier : "
    f"{slowest_person} "
    f"({average_response[slowest_person] / 60:.1f} hrs)"
)


print()
print("LONGEST SILENT PERIODS")

for person in sorted(
    people,
    key=lambda x: silent_days[x],
    reverse=True
):

    print(
        f"{person:<10} "
        f"{silent_days[person]} days"
    )


print()
print("PERSONALITY ARCHETYPES")

for person in people:

    print(
        f"{person:<10} → "
        f"{archetypes[person]}"
    )


print()
print("=" * 65)
print("Generated by GroupDNA")
print("Built with Python + NumPy")
print("=" * 65)


# 11. Project Constraints

According to the project brief, the implementation should use Python fundamentals, NumPy, file reading, `datetime`, and string methods.

The following are not allowed for the required implementation:
- pandas
- matplotlib / seaborn / plotly
- `collections.Counter`
- `collections.defaultdict`
- regular expressions (`re`)
- pre-built WhatsApp analyzer libraries
- AI / ML libraries

The activity heatmap is intentionally text-based rather than plotted with a visualization library.


# 12. Expected Outcome

After running the notebook with `hostel_bois.txt`, the notebook should generate:
- A parsed-message count
- Group overview statistics
- Busiest day and hour
- Activity heatmap
- Top words
- Response patterns
- Silent periods
- Personality archetypes
- A final formatted GroupDNA report


# 13. Reflection

### What was the hardest part?
Write about the part of parsing, analysis, NumPy, or archetype detection that required the most effort.

### What would I do differently?
Mention any changes you would make to improve the parser, analysis, code structure, or report formatting.

### My archetype
If you run the project on your own WhatsApp chat as the optional bonus, record your archetype here.

**Note:** Do not upload or publicly share your private WhatsApp chat. The project brief recommends sharing only an output screenshot after getting permission from the group members.
